# Analysis - the generator

A set of odor tokens goes in, a molecule comes out. This reads the committed evaluations in
`output/odor/` and draws their numbers; it runs no model.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SPLITS = ("validation", "test", "dream")
evaluations = {split: json.loads((ROOT / f"output/odor/evaluation_{split}.json").read_text()) for split in SPLITS}
print("loaded", {split: evaluations[split]["conditioning"]["molecules"] for split in SPLITS}, "molecules per split")

## 1. How probable the real molecules are

The generator is scored by the loss of writing a held-out molecule when it is told that molecule's own odor tokens,
in nats per molecule token. The request itself is not scored: the loss is masked from the first molecule token.
The gain is the loss without a request minus the loss with one.

In [ ]:
rows = []
for split in SPLITS:
    c = evaluations[split]["conditioning"]
    gain = c["gain_over_no_odor_tokens"]
    rows.append(
        {
            "split": split,
            "molecules": c["molecules"],
            "loss": round(c["loss_with_own_odor_tokens"], 4),
            "perplexity": round(c["perplexity_with_own_odor_tokens"], 2),
            "gain": f"{gain['mean']:+.3f} ± {gain['interval_95']:.3f}",
            "helped": f"{gain['helped']:.0%}",
        }
    )
pd.DataFrame(rows).set_index("split")

## 2. The gain is a per-molecule effect

An average gain can be carried by a few molecules. This is the distribution behind it on the test molecules: the
loss with the molecule's own odor tokens against the loss with none, one point per molecule.

In [ ]:
per = evaluations["test"]["conditioning"]["per_molecule"]
own = np.array(per["own"])
none = np.array(per["none"])
gain = none - own

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.scatter(none, own, s=9, alpha=0.5, color="#4472C4")
limits = [min(none.min(), own.min()), max(none.max(), own.max())]
ax.plot(limits, limits, color="#C00000", linewidth=1, linestyle="--", label="no change")
ax.set_xlabel("loss with no request")
ax.set_ylabel("loss with the molecule's own odor tokens")
ax.set_title(f"{len(own)} test molecules, {np.mean(gain > 0):.0%} helped")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"median gain {np.median(gain):+.3f}, mean {gain.mean():+.3f}, 5th-95th {np.percentile(gain, 5):+.3f} to {np.percentile(gain, 95):+.3f}")

### The gain as a distribution

The scatter shows the pairing; this shows how much of the distribution is on the right side of zero.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.hist(gain, bins=40, color="#4472C4")
ax.axvline(0, color="#C00000", linestyle="--", linewidth=1, label="no change")
ax.axvline(gain.mean(), color="#1F3864", linestyle="--", linewidth=1, label=f"mean {gain.mean():+.3f}")
ax.set_xlabel("loss with no request minus loss with the molecule's own tokens")
ax.set_ylabel("molecules")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print(f"{np.mean(gain > 0):.0%} of molecules helped, {np.mean(gain > 0.1):.0%} by more than 0.1 nats")

## 3. What it wrote

200 molecules for each of seven requests: six sets of odor tokens and one empty request, which is what the model
writes when asked for nothing.

`novel` counts distinct strings that appear in neither training set. The column beside it, how close the nearest
training molecule is, needs a generation run with the current code and reads `n/a` here.

In [ ]:
generation = evaluations["test"]["generation"]
table = pd.DataFrame(
    [
        {
            "request": request,
            "valid": round(found["valid"], 2),
            "unique": round(found["unique"], 2),
            "novel": round(found["novel"], 2),
            "median weight": round(found["weight_median"]),
            "over 350": f"{found['weight_over_350']:.1%}",
        }
        for request, found in generation.items()
    ]
).set_index("request")
table

In [ ]:
for request in ("fruity", "musk", "(nothing asked for)"):
    print(request)
    for smiles in generation[request]["examples"]:
        print("   ", smiles)

## Reading.

The loss is 0.7112 on validation, 0.7124 on test and 0.6972 on DREAM, with a perplexity of about 2: the model is
as unsure as choosing between two molecule tokens. The gain is real and it is not a handful of molecules. On test it is +0.212 nats with an interval of ±0.020, 90
percent of molecules are helped and 82 percent by more than 0.1 nats; on DREAM it is +0.298 and 97 percent. The extra 0.086 nats of gain on DREAM are worth noticing: those molecules come from a different
collection, and the model finds them easier to place once told their odor tokens.

What it writes is valid without exception, which is what SELFIES buys: 1.00 on every request for every split.
`unique` runs from 0.89 to 1.00 and `novel` from 0.70 to 0.86, so the sampler is not repeating one molecule in
different words. The empty request is the exception in the other direction: it is the most varied and the least
novel, 0.99 against 0.70 to 0.86, because with nothing asked for the model falls back on the pretraining
distribution. Its median weight, 271 Da, is the largest of the seven and the only one where 6 percent of the
molecules are over 350 Da. That is exactly what replay teaches the empty request to do.

The three splits give the same generation table, and that is not a bug: the sample depends on the requests and the
seed, not on which molecules were held out. It is reported once for that reason.

## Verdict

The request is read and it is worth +0.2 to +0.3 nats per molecule token, with the effect present on nine in ten
molecules. The output is valid, varied and mostly new. What this cannot say is whether any of it smells of what was
asked: nobody has smelled it, and no score here pretends otherwise.
